In [28]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [2]:
train_df = pd.read_parquet('../data/processed/splits/train.parquet')
train_df.head()

,user_id,movie_id,rating,timestamp,user_idx,item_idx,pos,len
0,1,3186,4,978300019,0,2969,0,53
1,1,1270,5,978300055,0,1178,1,53
2,1,1721,4,978300055,0,1574,2,53
3,1,1022,5,978300055,0,957,3,53
4,1,2340,3,978300103,0,2147,4,53


In [3]:
users_df = pd.read_csv(
    '../data/raw/ml-1m/users.dat', 
    engine='python',
    sep='::', 
    names=['user_id', 'gender', 'age', 'occupation', 'zip']
)
users_df.head()

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [4]:
train_df = train_df.merge(users_df, how='left', on='user_id')
train_df.head()

,user_id,movie_id,rating,timestamp,user_idx,item_idx,pos,len,gender,age,occupation,zip
0,1,3186,4,978300019,0,2969,0,53,F,1,10,48067
1,1,1270,5,978300055,0,1178,1,53,F,1,10,48067
2,1,1721,4,978300055,0,1574,2,53,F,1,10,48067
3,1,1022,5,978300055,0,957,3,53,F,1,10,48067
4,1,2340,3,978300103,0,2147,4,53,F,1,10,48067


In [5]:
train_df.isna().sum()

user_id       0
movie_id      0
rating        0
timestamp     0
user_idx      0
item_idx      0
pos           0
len           0
gender        0
age           0
occupation    0
zip           0
dtype: int64

In [6]:
SIDs_V1 = np.load('../data/processed/SIDs_V1.npy')
films = pd.read_parquet('../data/processed/item_features/item_meta.parquet')

In [7]:
COL_USER = 'user_idx'
COL_ITEM = 'item_idx'
COL_RATING = 'rating'

COL_TITLE = 'title'
COL_YEARS = 'years'
COL_GENRES = 'genres'

COL_GENDER = 'gender'
COL_AGE = 'age'
COL_OCCUPATION = 'occupation'

BOS = '<bos>'
EOS = '<eos>'
PAD = '<pad>'
UNK = '<unk>'

TOK_USER_OPEN  = '<user>'
TOK_USER_CLOSE = '</user>'
TOK_HIST       = '<hist>'
TOK_E_OPEN     = '<e>'
TOK_E_CLOSE    = '</e>'

TOK_TITLE_OPEN = '<title>'
TOK_TITLE_CLOSE = '</title>'
TOK_YEAR_OPEN = '<year>'
TOK_YEAR_CLOSE = '</year>'
TOK_GENRES_OPEN = '<genres>'
TOK_GENRES_CLOSE = '</genres>'


def get_sid_token(sid):
    return [f'<sid_{i}_{j}>' for i, j in enumerate(sid)]

def get_rating_token(rating):
    return f'<rat_{int(rating)}>'

def get_user_token(gender, age, occupation):
    return [f'<gen_{gender}>', f'<age_{int(age)}>', f'<occ_{occupation}>']

In [8]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.add_special_tokens({
    'bos_token' : BOS,
    'eos_token' : EOS,
    'pad_token' : PAD,
    'unk_token' : UNK
})

train_items = train_df[COL_ITEM].unique()

extra = set([TOK_USER_OPEN, TOK_USER_CLOSE, TOK_HIST, TOK_E_OPEN, 
             TOK_E_CLOSE, TOK_TITLE_OPEN, TOK_TITLE_CLOSE, TOK_YEAR_OPEN, 
             TOK_YEAR_CLOSE, TOK_GENRES_OPEN, TOK_GENRES_CLOSE
             
])

In [9]:
users = train_df[[COL_USER, COL_GENDER, COL_AGE, COL_OCCUPATION]].drop_duplicates().copy()
users.isna().sum()

user_idx      0
gender        0
age           0
occupation    0
dtype: int64

In [10]:
extra.update([get_rating_token(i) for i in range(1, 6)])
extra.update([f'<gen_{gender}>' for gender in users[COL_GENDER].unique().tolist()])
extra.update([f'<age_{int(age)}>' for age in users[COL_AGE].unique().tolist()])
extra.update([f'<occ_{int(occupation)}>' for occupation in users[COL_OCCUPATION].unique().tolist()])

for item_idx in films[COL_ITEM].unique().tolist():
    extra.update(get_sid_token(SIDs_V1[item_idx, :]))

In [11]:
tokenizer.add_tokens(sorted(extra))

577

### User behavior training data

In [12]:
train_df.head()

,user_id,movie_id,rating,timestamp,user_idx,item_idx,pos,len,gender,age,occupation,zip
0,1,3186,4,978300019,0,2969,0,53,F,1,10,48067
1,1,1270,5,978300055,0,1178,1,53,F,1,10,48067
2,1,1721,4,978300055,0,1574,2,53,F,1,10,48067
3,1,1022,5,978300055,0,957,3,53,F,1,10,48067
4,1,2340,3,978300103,0,2147,4,53,F,1,10,48067


In [13]:
behavior_training_data = []
counter = 0

for user_id, user_group in train_df.groupby(COL_USER):
    sequence = [BOS, TOK_USER_OPEN]

    first_row = user_group.iloc[0]
    personal_data = first_row[[COL_GENDER, COL_AGE, COL_OCCUPATION]].tolist()
    user_token = get_user_token(*personal_data)
    sequence.extend([*user_token, TOK_USER_CLOSE, TOK_HIST])
    
    user_group = user_group.iloc[-125:] # last 125 movies
    
    for _, row in user_group.iterrows():
        item_idx = row[COL_ITEM]
        rating = row[COL_RATING]

        sid_token = get_sid_token(SIDs_V1[item_idx, :])
        rating_token = get_rating_token(rating)

        sequence.extend([TOK_E_OPEN, *sid_token, rating_token, TOK_E_CLOSE])

    
    sequence.append(EOS)
    ids = tokenizer.convert_tokens_to_ids(sequence)
    if len(ids) > 1024:
        counter += 1
    behavior_training_data.append(ids)
    
counter

0

In [14]:
counter

0

In [15]:
behavior_training_data[0][:24]

[50257,
 50836,
 50274,
 50267,
 50279,
 50264,
 50277,
 50273,
 50326,
 50378,
 50613,
 50728,
 50764,
 50302,
 50261,
 50273,
 50323,
 50348,
 50551,
 50732,
 50826,
 50303,
 50261,
 50273]

### Fil metadata corpus

#### SID + movie title

In [16]:
films.head()

,movie_id,item_idx,title,years,genres
0,1,0,Toy Story,1995,"[Animation, Children's, Comedy]"
1,2,1,Jumanji,1995,"[Adventure, Children's, Fantasy]"
2,3,2,Grumpier Old Men,1995,"[Comedy, Romance]"
3,4,3,Waiting to Exhale,1995,"[Comedy, Drama]"
4,5,4,Father of the Bride Part II,1995,[Comedy]


In [17]:
movie_metadata_title = []

for _, row in films.iterrows():
    item_idx = row[COL_ITEM]
    title = row[COL_TITLE].rstrip()

    sid_tokens = get_sid_token(SIDs_V1[item_idx, :]) 

    text = (
        f'{BOS}{TOK_TITLE_OPEN}'
        f'Movie {"".join(sid_tokens)} has title: {title}'
        f'{TOK_TITLE_CLOSE}{EOS}'
    )

    ids = tokenizer.encode(text, add_special_tokens=False)
    movie_metadata_title.append(ids)

In [18]:
movie_metadata_title[0]

[50257,
 50835,
 25097,
 220,
 50322,
 50334,
 50576,
 50703,
 50814,
 468,
 3670,
 25,
 10977,
 8362,
 50263,
 50258]

#### SID + film genres

In [19]:
movie_metadata_genre  = []

for _, row in films.iterrows():
    sequence = [BOS, TOK_GENRES_OPEN]

    item_idx = row[COL_ITEM]
    genres = row[COL_GENRES]
    genres_with_sep = []
    sep = ','
    for j, g in enumerate(genres):
        if j > 0:
            genres_with_sep.append(sep)
        genres_with_sep.append(g)

    sid_token = get_sid_token(SIDs_V1[item_idx, :])

    text = (
        f'{BOS}{TOK_GENRES_OPEN}'
        f'The genres in movie {"".join(sid_token)} are:'
        f'{"".join(genres_with_sep)}'
        f'{TOK_GENRES_CLOSE}{EOS}'
        
    )

    ids = tokenizer.encode(text, add_special_tokens=False)
    movie_metadata_genre.append(ids)

In [20]:
movie_metadata_genre[0]

[50257,
 50276,
 464,
 27962,
 287,
 3807,
 220,
 50322,
 50334,
 50576,
 50703,
 50814,
 389,
 25,
 39520,
 11,
 26829,
 338,
 11,
 5377,
 4716,
 50262,
 50258]

#### SID + film release date

In [21]:
movie_metadata_year = []

for _, row in films.iterrows():
    sequence = [BOS, TOK_YEAR_OPEN]

    item_idx = row[COL_ITEM]
    year = row[COL_YEARS]
    sid_token = get_sid_token(SIDs_V1[item_idx, :])

    text = (
        f'{BOS}{TOK_YEAR_OPEN}'
        f'The movie {"".join(sid_token)} was released in {str(year)}'
        f'{TOK_YEAR_CLOSE}{EOS}'
    )

    ids = tokenizer.encode(text, add_special_tokens=False)
    movie_metadata_year.append(ids)

In [22]:
movie_metadata_year[0]

[50257,
 50837,
 464,
 3807,
 220,
 50322,
 50334,
 50576,
 50703,
 50814,
 373,
 2716,
 287,
 8735,
 50265,
 50258]

In [29]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer.save_pretrained(out_dir / 'tokenizer')

token_spec = {
    'special_tokens': {
        'bos': BOS, 'eos': EOS, 'pad': PAD, 'unk': UNK
    },
    'schema_tokens': [
        TOK_USER_OPEN, TOK_USER_CLOSE, TOK_HIST, TOK_E_OPEN, TOK_E_CLOSE,
        TOK_TITLE_OPEN, TOK_TITLE_CLOSE, TOK_YEAR_OPEN, TOK_YEAR_CLOSE,
        TOK_GENRES_OPEN, TOK_GENRES_CLOSE
    ],
    'notes': {
        'behavior_last_k': 125,
        'meta_title_template': 'Movie <sid...> has title: {title}',
        'meta_genre_template': 'The genres in movie <sid...> are: {genres}',
        'meta_year_template': 'The movie <sid...> was released in {year}'
    }
}

with open(out_dir / 'token_spec.json', 'w', encoding='utf-8') as f:
    json.dump(token_spec, f, ensure_ascii=False, indent=2)

np.save(out_dir / 'SIDs_V1.npy', SIDs_V1)

def save_ds(name, ids):
    ds = Dataset.from_dict({'input_ids' : ids})
    dr = out_dir / 'data' / name
    ds.save_to_disk(dr)

save_ds('behavior', behavior_training_data)
save_ds('meta_title', movie_metadata_title)
save_ds('meta_genre', movie_metadata_genre)
save_ds('meta_year', movie_metadata_year)

Saving the dataset (0/1 shards):   0%|          | 0/6040 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3706 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3706 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3706 [00:00<?, ? examples/s]